### Bridge NCF Visualization

In [ ]:
import glob
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

import dask
from dask.diagnostics import ProgressBar
from IPython.display import HTML
from tqdm import tqdm

sys.path.append('..')
from src.utils import parse_ncf_stack_filename
from src.disp import dispersion_curve
from src.ncf import get_vs_number, export_targeted_ncfs, fast_batch_processor, split_ncf_sides_for_dispersion
from src.plot import animate_ncf_section_mesh, animate_directional_ncf_section_mesh, animate_preprocessed_ncf, animate_preprocessed_fk, animate_pipeline, animate_fv

import matplotlib as mpl
mpl.rcParams["animation.html"] = "jshtml"
mpl.rcParams["animation.embed_limit"] = 500.0 

old_fig_dpi = plt.rcParams['figure.dpi']
old_save_dpi = plt.rcParams['savefig.dpi']

#### 1. Read NCFs

In [ ]:
# Choose dataset
pattern_conventional_3d = "../data/ncf_stacks_bridge/3d/20250724_cc_*_3d_conventional.npy"
pattern_v1_3d = "../data/ncf_stacks_bridge/3d/20250724_cc_*_3d_v1.npy"

files_conventional = sorted(glob.glob(pattern_conventional_3d), key=get_vs_number)
files_v1 = sorted(glob.glob(pattern_v1_3d), key=get_vs_number)

print("Matched files:", len(files_conventional))
print(f"First 3 files: {[os.path.basename(f) for f in files_conventional[:3]]}")
print(f"Last 3 files: {[os.path.basename(f) for f in files_conventional[-3:]]}")

print("Matched files:", len(files_v1))
print(f"First 3 files: {[os.path.basename(f) for f in files_v1[:3]]}")
print(f"Last 3 files: {[os.path.basename(f) for f in files_v1[-3:]]}")

In [ ]:
# 1. Load ONE file to infer axes 
# We only need one representative NCF to build lag_axis & distance_axis
ncf0_path = files_conventional[0]
ncf0 = np.load(ncf0_path)

# Parse metadata (date, vs, window, mode)
date0, vs0, window0, mode0 = parse_ncf_stack_filename(ncf0_path)
print("Example parsed:", date0, vs0, window0, mode0)

In [ ]:
# 2. Define parameters
fs = 250.0      # Sampling frequency in Hz
dt = 1 / fs     # sampling rate in sec

max_lag = 1     # seconds
npts_lag = int(max_lag * fs)

# Cable configuration (used only to create distance_axis length n_rec)
first_chan = 0
last_chan = 420
dx = 2.45
channels = np.arange(first_chan, last_chan + 1)

n_rec, n_lags = ncf0.shape

# Lag axis spans from –max_lag ... +max_lag
lag_axis = np.linspace(-max_lag, +max_lag, n_lags)

distance_axis = (channels - channels[0]) * dx

print(f"NCF shape: {ncf0.shape}, Lag axis: {lag_axis.shape}, Distance axis: {distance_axis.shape}")

In [ ]:
try:
    # Force global DPI down just for this rendering phase
    plt.rcParams.update({
        'figure.dpi': 120,        
        'savefig.dpi': 120,
        'axes.titlesize': 14,
    })

    # Generate the animation
    ani = animate_ncf_section_mesh(
        pattern_conventional_3d,
        lag_axis=lag_axis,
        distance_axis=distance_axis,
        mode="all",
        unit="m",
        pclip=99,
        dx=dx,
        range_m=200,
        clip_lim=True,
        interval_ms=150,
        figsize=(6, 6)   
    )
    
    # Render the HTML while the low-DPI settings are still active
    output = HTML(ani.to_jshtml())

finally:
    # Safely restore high-res settings for other plots
    plt.rcParams.update({
        'figure.dpi': old_fig_dpi,
        'savefig.dpi': old_save_dpi,
    })

# Display the output
output

#### 2. Spatial-Temporal Swapping/Folding

In [ ]:
try:
    plt.rcParams.update({
        'figure.dpi': 120,        
        'savefig.dpi': 120,
        'axes.titlesize': 14,
    })

    ani_dir = animate_directional_ncf_section_mesh(
    pattern_conventional_3d,
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    target="s1",              # Look at the 's1' directional fold
    unit="m",
    pclip=99,
    dx=dx,           
    range_m=100,              # View +/- 100m around the virtual source
    view_side="both",         # Optional: Only look right of the source
    pos_offset=0.0,           # Exclude the first 100m (near-field artifacts)
    vs_start=90,              # Start
    vs_end=340,               # End
    max_lag=None,             # Crop the Y-axis 
)
    
    output = HTML(ani_dir.to_jshtml())

finally:
    plt.rcParams.update({
        'figure.dpi': old_fig_dpi,
        'savefig.dpi': old_save_dpi,
    })

output

#### 3. Prepare NCFs Before Processing

In [ ]:
output_folder = "../data/ncf_pre_bridge"
export_targeted_ncfs(
    pattern_conventional_3d,
    out_dir=output_folder,
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    target="s1",                 
    dx=dx,
    range_m=100,              
    view_side="both",          
    pos_offset=0.0,            
    vs_start=90,              
    vs_end=340,               
    max_lag=None,                  
    taper_alpha=0.1               
)

export_targeted_ncfs(
    pattern_conventional_3d,
    out_dir=output_folder,
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    target="s2",                 
    dx=dx,
    range_m=100,              
    view_side="both",          
    pos_offset=0.0,            
    vs_start=90,              
    vs_end=340,               
    max_lag=None,                  
    taper_alpha=0.1               
)

export_targeted_ncfs(
    pattern_v1_3d,
    out_dir=output_folder,
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    target="s1",                 
    dx=dx,
    range_m=100,              
    view_side="both",          
    pos_offset=0.0,            
    vs_start=90,              
    vs_end=340,               
    max_lag=None,                  
    taper_alpha=0.1               
)

export_targeted_ncfs(
    pattern_v1_3d,
    out_dir=output_folder,
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    target="s2",                 
    dx=dx,
    range_m=100,              
    view_side="both",          
    pos_offset=0.0,            
    vs_start=90,              
    vs_end=340,               
    max_lag=None,                  
    taper_alpha=0.1               
)

In [ ]:
# Choose dataset
pattern_conventional_3d_s1 = "../data/ncf_pre_bridge/20250724_cc_*_3d_conventional_s1.npz"
pattern_conventional_3d_s2 = "../data/ncf_pre_bridge/20250724_cc_*_3d_conventional_s2.npz"

pattern_v1_3d_s1 = "../data/ncf_pre_bridge/20250724_cc_*_3d_v1_s1.npz"
pattern_v1_3d_s2 = "../data/ncf_pre_bridge/20250724_cc_*_3d_v1_s2.npz"

files_s1_conventional = sorted(glob.glob(pattern_conventional_3d_s1), key=get_vs_number)
files_s2_conventional = sorted(glob.glob(pattern_conventional_3d_s2), key=get_vs_number)
print("Matched files (s1):", len(files_s1_conventional))
print("Matched files (s2):", len(files_s2_conventional))
print("First 3 (s1):", files_s1_conventional[:3])
print("First 3 (s2):", files_s2_conventional[:3])

#### 4. Display Pre-Processed NCFs

In [ ]:
try:
    plt.rcParams.update({
        'figure.dpi': 120,        
        'savefig.dpi': 120,
        'axes.titlesize': 14,
    })

    ani_pre = animate_preprocessed_ncf(
    files=files_s1_conventional,           
    unit="m",
    pclip=99,
    range_m=100.0,             
    view_side="both",          
    pos_offset=0.0,           
)
    
    output = HTML(ani_pre.to_jshtml())

finally:
    plt.rcParams.update({
        'figure.dpi': old_fig_dpi,
        'savefig.dpi': old_save_dpi,
    })

output

#### 5. Display f-k NCFs

In [ ]:
try:
    plt.rcParams.update({
        'figure.dpi': 120,        
        'savefig.dpi': 120,
        'axes.titlesize': 14,
    })

    ani_fk = animate_preprocessed_fk(
    files=files_s1_conventional,
    unit="m",
    pclip=99.5,                
    view_side="both",
    pos_offset=10.0,
    klim=None,              
    figsize=(9, 6),                           
    vmin=200,                  
    vmax=4000,                  
    interval_ms=10
)
    
    output = HTML(ani_fk.to_jshtml())

finally:
    plt.rcParams.update({
        'figure.dpi': old_fig_dpi,
        'savefig.dpi': old_save_dpi,
    })

output

#### 6. NCF Processing: fk + Mute + Taper

In [ ]:
data_dir = "../data/ncf_pre_bridge"
processed_out_dir = "../data/ncf_fk_bridge/"
os.makedirs(processed_out_dir, exist_ok=True)

pattern_conventional_3d_s1 = os.path.join(data_dir, "20250724_cc*_3d_conventional_s1.npz")
pattern_conventional_3d_s2 = os.path.join(data_dir, "20250724_cc*_3d_conventional_s2.npz")
pattern_v1_3d_s1 = os.path.join(data_dir, "20250724_cc*_3d_v1_s1.npz")
pattern_v1_3d_s2 = os.path.join(data_dir, "20250724_cc*_3d_v1_s2.npz")

files_s1_conventional = sorted(glob.glob(pattern_conventional_3d_s1), key=get_vs_number)
files_s2_conventional = sorted(glob.glob(pattern_conventional_3d_s2), key=get_vs_number)
files_s1_v1 = sorted(glob.glob(pattern_v1_3d_s1), key=get_vs_number)
files_s2_v1 = sorted(glob.glob(pattern_v1_3d_s2), key=get_vs_number)

processing_queue = [
    (files_s1_conventional, "s1_conventional_bridge.mp4", "S1"),
    (files_s2_conventional, "s2_conventional_bridge.mp4", "S2"), 
    (files_s1_v1, "s1_v1_bridge.mp4", "S1"),
    (files_s2_v1, "s2_v1_bridge.mp4", "S2"), 
]

for file_list, mp4_name, label in processing_queue:
    if not file_list:
        print(f"Skipping {label}: No files matched the pattern.")
        continue
        
    print(f"\n{'='*50}")
    print(f"🚀 STARTING PIPELINE: {label} ({len(file_list)} files)")
    print(f"{'='*50}")
    print(f"First 3: {[os.path.basename(f) for f in file_list[:3]]}")
    print(f"Last 3:  {[os.path.basename(f) for f in file_list[-3:]]}")

    # 1. Generate the animation object 
    anim_plot = animate_pipeline(
        file_list=file_list, 
        out_dir=None, 
        vmin=200.0, 
        vmax=4000.0,
        vmax_time=1000.0,      # Time Group Velocity (Slopes the top mute down tighter)
        pos_offset=2.0,
        inner_taper=2.0,
        range_m=100.0,
        pclip=99.0,
        interval=1000, 
        buffer_start_s=0.02,
        buffer_end_s=0.20,
        top_flat_m=0.5
    )

    # 2. Save the MP4 
    print(f"\nSaving video to [{mp4_name}]...")
    print("This will take a few minutes. Watch the progress bar below:")
    
    anim_plot.save(
        mp4_name, 
        fps=2, 
        dpi=150, 
        extra_args=['-vcodec', 'libx264'] 
    )

    print(f"\n✅ Video Render Complete for {label}!")

print("\n🎉 All validation datasets rendered successfully.")

In [ ]:
# Process and save NCFs (fk + mute + taper)
# Define Paths
data_dir = "../data/ncf_pre_bridge"
processed_out_dir = "../data/ncf_fk_bridge"

pattern_conventional_s1 = os.path.join(data_dir, "20250724_cc*_3d_conventional_s1.npz")
pattern_conventional_s2 = os.path.join(data_dir, "20250724_cc*_3d_conventional_s2.npz")
pattern_v1_s1 = os.path.join(data_dir, "20250724_cc*_3d_v1_s1.npz")
pattern_v1_s2 = os.path.join(data_dir, "20250724_cc*_3d_v1_s2.npz")

files_s1_conventional = sorted(glob.glob(pattern_conventional_s1), key=get_vs_number)
files_s2_conventional = sorted(glob.glob(pattern_conventional_s2), key=get_vs_number)
files_s1_v1 = sorted(glob.glob(pattern_v1_s1), key=get_vs_number)
files_s2_v1 = sorted(glob.glob(pattern_v1_s2), key=get_vs_number)

print(f"Found {len(files_s1_conventional)} S1 files and {len(files_s2_conventional)} S2 files.")

if files_s1_conventional:
    print("\n--- Processing S1 ---")
    fast_batch_processor(files_s1_conventional, processed_out_dir, vmin=200.0, vmax=4000.0, vmax_time=1000.0, pos_offset=2.0, inner_taper=2.0, range_m=100.0, sigma=1.0, buffer_start_s=0.02, buffer_end_s=0.20, top_flat_m=0.5)

if files_s2_conventional:
    print("\n--- Processing S2 ---")
    fast_batch_processor(files_s2_conventional, processed_out_dir, vmin=200.0, vmax=4000.0, vmax_time=1000.0, pos_offset=2.0, inner_taper=2.0, range_m=100.0, sigma=1.0, buffer_start_s=0.02, buffer_end_s=0.20, top_flat_m=0.5)

if files_s1_v1:
    print("\n--- Processing S1 ---")
    fast_batch_processor(files_s1_v1, processed_out_dir, vmin=200.0, vmax=4000.0, vmax_time=1000.0, pos_offset=2.0, inner_taper=2.0, range_m=100.0, sigma=1.0, buffer_start_s=0.02, buffer_end_s=0.20, top_flat_m=0.5)

if files_s2_v1:
    print("\n--- Processing S2 ---")
    fast_batch_processor(files_s2_v1, processed_out_dir, vmin=200.0, vmax=4000.0, vmax_time=1000.0, pos_offset=2.0, inner_taper=2.0, range_m=100.0, sigma=1.0, buffer_start_s=0.02, buffer_end_s=0.20, top_flat_m=0.5)

print(f"\n✅ All files successfully processed and saved to {processed_out_dir}")

#### 7. Display Processed NCFs

In [ ]:
pattern_conventional_3d_fk_s1 = "../data/ncf_fk_bridge/20250724_cc*_3d_conventional_s1_fk_*_*.npz"

files_s1_conventional = sorted(glob.glob(pattern_conventional_3d_fk_s1), key=get_vs_number)
print(f"Matched files (S1): {len(files_s1_conventional)}")

if files_s1_conventional:
    print(f"First 3 (S1): {[os.path.basename(f) for f in files_s1_conventional[:3]]}")
if files_s1_conventional:
    try:
        plt.rcParams.update({
            'figure.dpi': 100,        
            'savefig.dpi': 100,
            'axes.titlesize': 14,
        })

        ani_pre = animate_preprocessed_ncf(
        files=files_s1_conventional,           
        unit="m",
        pclip=99,
        range_m=100.0,             
        view_side="both",          
        pos_offset=0.0,           
        )
        
        output = HTML(ani_pre.to_jshtml())

    finally:
        plt.rcParams.update({
            'figure.dpi': old_fig_dpi,
            'savefig.dpi': old_save_dpi,
        })
output

#### 8. Choose Good VS

In [ ]:
# 1. Define Paths and Patterns
data_dir = "../data/ncf_fk_bridge"
out_dir = "../data/ncf_disp_bridge"
os.makedirs(out_dir, exist_ok=True)

pattern_conventional_s1 = os.path.join(data_dir, "20250724_cc*_3d_conventional_s1_fk_*_*.npz")
pattern_conventional_s2 = os.path.join(data_dir, "20250724_cc_*_3d_conventional_s2_fk_*_*.npz")
pattern_v1_s1 = os.path.join(data_dir, "20250724_cc*_3d_v1_s1_fk_*_*.npz")
pattern_v1_s2 = os.path.join(data_dir, "20250724_cc_*_3d_v1_s2_fk_*_*.npz")

files_s1_conventional = sorted(glob.glob(pattern_conventional_s1), key=get_vs_number)
files_s2_conventional = sorted(glob.glob(pattern_conventional_s2), key=get_vs_number)
files_s1_v1 = sorted(glob.glob(pattern_v1_s1), key=get_vs_number)
files_s2_v1 = sorted(glob.glob(pattern_v1_s2), key=get_vs_number)

# 2. Batch Execution Queue
# Add or remove datasets from this list to control what runs
processing_queue = [
    (files_s1_conventional, "S1"),
    (files_s2_conventional, "S2"), 
    (files_s1_v1, "S1"),
    (files_s2_v1, "S2"), 
]

total_saved = 0

for file_list, label in processing_queue:
    if not file_list:
        print(f"⏭️ Skipping {label}: No files matched the pattern.")
        continue

    print(f"\n{'='*50}")
    print(f"🚀 SPLITTING SIDES: {label} ({len(file_list)} files)")
    print(f"{'='*50}")
    print(f"First 3: {[os.path.basename(f) for f in file_list[:3]]}")

    # Build Dask Task Graph
    tasks = [split_ncf_sides_for_dispersion(path, out_dir) for path in file_list]

    print(f"\nDispatching {len(tasks)} files to Dask workers...")

    # Execute with Progress Bar
    with ProgressBar():
        results = dask.compute(*tasks)

    # Cleanly flatten the list of lists and remove Nones
    final_paths = [path for res in results if res is not None for path in res]
    total_saved += len(final_paths)

    print(f"✅ Finished {label}. Saved {len(final_paths)} directional files.")

print(f"\n🎉 Process Complete! Total directional files saved to {out_dir}: {total_saved}")

#### 9. Construct Dispersion Images

In [ ]:
# Frequency-Velocity Panel Parameters
fv_kwargs = {
    "vmin": 200.0,
    "vmax": 4000.0,
    "dv": 2.0,       # 2 m/s resolution  
    "fmin": 1,     
    "fmax": 125,     
    "normalize": True,
    "device": torch.device("cpu") 
}

input_dir = "../data/ncf_disp_bridge"
fv_out_dir = "../results/fv_panels_bridge"
os.makedirs(fv_out_dir, exist_ok=True)

In [ ]:
# Grab the newly split directional .npz files and sort them spatially
processed_files = sorted(glob.glob(f"{input_dir}/*.npz"), key=get_vs_number)
fv_results = {}

# Batch Compute f-v Panels
for fpath in tqdm(processed_files, desc="Computing & Saving f-v Panels"):
    
    # Load the pristine array data
    archive = np.load(fpath)
    data = archive["data"]
    offset = archive["offset"]
    lag = archive["lag"]
    side = str(archive["side"])
    
    # Convert all offsets to absolute distance. 
    # This ensures left-side (negative) traces scan positive velocities correctly.
    dist_abs = np.abs(offset)
    
    # Compute f-v panel (Phase-Shift Method)
    fv, f_axis, v_axis = dispersion_curve(
        data=data, 
        offset=dist_abs, 
        t=lag, 
        **fv_kwargs
    )
    
    fname = os.path.basename(fpath)
    save_name = fname.replace(".npz", "_fv.npz")
    save_path = os.path.join(fv_out_dir, save_name)
    
    # Move tensors to CPU/Numpy for storage compatibility
    fv_data_cpu = fv.cpu().numpy() if hasattr(fv, 'cpu') else fv
    f_axis_cpu = f_axis.cpu().numpy() if hasattr(f_axis, 'cpu') else f_axis
    v_axis_cpu = v_axis.cpu().numpy() if hasattr(v_axis, 'cpu') else v_axis
    
    np.savez_compressed(
        save_path,
        fv=fv_data_cpu,
        f_axis=f_axis_cpu,
        v_axis=v_axis_cpu,
        side=side
    )
    
    fv_results[fname] = {"fv": fv_data_cpu, "side": side}

print(f"Successfully saved {len(processed_files)} panels to {fv_out_dir}")


In [ ]:
fv_out_dir = "../results/fv_panels_bridge"
all_saved_fv = sorted(glob.glob(f"{fv_out_dir}/*_fv.npz"))

right_side_files_s1_conventional = [
    f for f in all_saved_fv 
    if "_right" in f and "_s1" in f and "conventional" in f
]

try:
    plt.rcParams.update({
        'figure.dpi': 120,        
        'savefig.dpi': 120,
        'axes.titlesize': 14,
    })

    ani_smooth = animate_fv(
    fv_files=right_side_files_s1_conventional, 
    xmin=1,    
    xmax=125,    
    ymin=200.0,   
    ymax=4000.0, 
    cmap="viridis",
    interval_ms=300, 
    figsize=(8, 6)
)
    
    output = HTML(ani_smooth.to_jshtml())

finally:
    plt.rcParams.update({
        'figure.dpi': old_fig_dpi,
        'savefig.dpi': old_save_dpi,
    })

output

**Convert notebook to html**
```bash
python3 -m jupyter nbconvert --to html ncf_bridge.ipynb
```